# 재정렬기(Qwen3.5 VLM) LoRA 파인튜닝 (Colab)

1단계(FG-CLIP)가 뽑은 top-K 후보를 VLM에 "이 그림이 설명과 맞나? Yes/No"로 물어 다시 정렬한다. chan_lee는 이 재정렬기를 **zero-shot**으로만 썼다. 여기서는 train 데이터로 재정렬기를 **LoRA 파인튜닝**하고 같은 val로 비교한다.

| 조건 | 재정렬기 | 학습 때 오답 | 답하는 질문 |
|---|---|---|---|
| **R0** | zero-shot (chan_lee 방식 그대로) | — | 기준선 |
| **R1** | LoRA FT | 같은 묶음 형제 이미지 + 1단계가 헷갈린 오답 | 재정렬기 파인튜닝 효과 |
| **R2** | LoRA FT | R1 + 자기 train HN 이미지(398장, 있으면 1개 고정) | HN 이미지가 재정렬 단계에서는 도움이 되나 (Q3) |

- 학습: 프롬프트마다 [정답 1 + 오답 3] 묶음, 묶음 안 Yes−No 점수에 softmax CE. 비전 인코더 고정, 언어 모델 Linear 층에만 LoRA. 추론은 쌍마다 독립 채점(chan_lee와 동일).
- 평가: chan_lee와 같은 묶음 val 500문항, 후보 풀 = val 원본 500 + val HN 102. 1단계 top-5 / top-10을 재정렬.
- 학습 데이터: train 3,500문항(val 묶음 제외). test는 쓰지 않는다.

**순서**: 1) Drive → 2) 설정 → 3) 설치 → 4) 데이터 → 5) 코드 → 6) 1단계 후보 → 7) R0 → 8) R1 → 9) R2 → 10) 비교 → 11) test 재정렬

- **test만 돌릴 때**: 1~5 실행(SMOKE=False) → 6의 두 번째 셀(함수 정의)은 건너뛰어도 됨 → 11로 바로 간다.

- 런타임: **GPU (A100 권장, L4 가능)**. 결과(1단계 후보, LoRA 어댑터, 점수)는 Drive `RERANK_DIR`에 저장되고, 이미 있으면 건너뛴다. **런타임이 끊기면 1~5를 다시 실행한 뒤 이어서** 돌리면 된다.
- **처음에는 `SMOKE = True`로 끝까지 한 번 돌려** 오류가 없는지 확인한다(몇 분). 확인되면 `False`로 바꾸고 1~5 다음 6부터 다시 실행한다. SMOKE 결과는 별도 폴더에 저장된다.
- 필요한 Drive 파일: `train_v1.tar`, `val_ids.txt`, `train_bundles.json`(레포 `finetune/`에 있음). 1단계 FG-CLIP은 `OUT_DIR/bundle_bval` → `OUT_DIR/ctrl` 순서로 찾고, 없으면 zero-shot `qihoo360/fg-clip-base`를 쓴다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 설정

In [ ]:
import os, glob, json, subprocess

def find_in_drive(name):
    hits = glob.glob(f"/content/drive/MyDrive/**/{name}", recursive=True)
    assert hits, f"MyDrive 에서 {name} 를 못 찾음 -> Drive 에 올리거나 경로를 직접 적으세요"
    return hits[0]

SMOKE = True            # True: 학습 64문항 / 평가 30문항으로 파이프라인만 점검. 확인 후 False

# --- 데이터 ---
TRAIN_TAR = ""          # 비우면 MyDrive 에서 train_v1.tar 자동 탐색
VAL_IDS_PATH = ""       # 비우면 val_ids.txt 자동 탐색
BUNDLES_PATH = ""       # 비우면 train_bundles.json 자동 탐색
OUT_DIR = "/content/drive/MyDrive/haikuthon_runs"
STAGE1 = ""             # 비우면 OUT_DIR/bundle_bval -> OUT_DIR/ctrl -> qihoo360/fg-clip-base 순서

# --- 재정렬기 ---
RERANKER = "Qwen/Qwen3.5-2B"
MAX_PIXELS = 448 * 448  # chan_lee 와 같음 (200,704)
KS = (5, 10)            # 재정렬 후보 수

# --- 학습 (R1/R2 공통) ---
N_NEG = 3               # 프롬프트당 오답 수 -> 묶음 크기 4
PROMPTS_PER_STEP = 2    # 한 번에 GPU 에 올리는 프롬프트 수 (시퀀스 2x4=8). OOM 이면 1
GRAD_ACCUM = 4          # 실효 배치 = 2x4 = 8 프롬프트
EPOCHS = 1
LR = 1e-4
LORA = {"r": 16, "alpha": 32, "dropout": 0.05}
SEED = 0

RERANK_DIR = f"{OUT_DIR}/rerank" + ("_smoke" if SMOKE else "")
MAX_PROMPTS = 64 if SMOKE else None
EVAL_LIMIT = 30 if SMOKE else None
os.makedirs(RERANK_DIR, exist_ok=True)

VAL_IDS_PATH = VAL_IDS_PATH or find_in_drive("val_ids.txt")
BUNDLES_PATH = BUNDLES_PATH or find_in_drive("train_bundles.json")
if not STAGE1:
    STAGE1 = next((d for d in (f"{OUT_DIR}/bundle_bval", f"{OUT_DIR}/ctrl")
                   if os.path.exists(f"{d}/model.safetensors")), "qihoo360/fg-clip-base")
print("STAGE1     =", STAGE1)
print("RERANK_DIR =", RERANK_DIR)
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 (런타임 유형을 GPU 로 바꾸세요)")

## 3. 설치

- 재정렬기(Qwen3.5)용 transformers 5.x를 시스템에 설치한다 (chan_lee `requirements.txt`와 같은 버전).
- 1단계 FG-CLIP 원격코드는 transformers 4.46.3이 필요해서 `/content/tf446`에 따로 설치하고, 6번에서 `PYTHONPATH`로 앞세워 별도 프로세스로 돌린다.
- `flash-linear-attention`: Qwen3.5의 linear attention CUDA 커널. 설치가 실패하면 느린 기본 구현으로 돌아간다(점수는 같음).

In [ ]:
!pip install -q "transformers==5.22.2" "peft==0.21.2" "accelerate==1.15.0"
!pip uninstall -y -q torchao   # Colab 기본 torchao 0.10 은 peft 와 충돌(>=0.16 요구). 재정렬기에는 필요 없음
!pip install -q --no-deps --target /content/tf446 "transformers==4.46.3" "tokenizers==0.20.3" "huggingface_hub==0.26.5" "einops==0.8.1"
!pip install -q flash-linear-attention || echo "flash-linear-attention 설치 실패 -> 기본 구현 사용"
import transformers, peft; print("transformers", transformers.__version__, "| peft", peft.__version__)

## 4. 데이터: Drive → /content 로 복사 후 압축 해제

In [ ]:
if not TRAIN_TAR:
    TRAIN_TAR = find_in_drive("train_v1.tar")
os.makedirs("/content/data", exist_ok=True)
if not glob.glob("/content/data/**/prompts.jsonl", recursive=True):
    !cp "{TRAIN_TAR}" /content/train_v1.tar
    !tar -xf /content/train_v1.tar -C /content/data
    !rm /content/train_v1.tar
PROMPTS = glob.glob("/content/data/**/prompts.jsonl", recursive=True)[0]
DATA_DIR = os.path.dirname(PROMPTS)
print("DATA_DIR =", DATA_DIR, "| png:", sum(1 for _ in glob.iglob(DATA_DIR + "/images/**/*.png", recursive=True)), "(4500 이어야 함)")

## 5. 코드 파일 생성 (레포 `finetune/rerank/`와 같은 내용)

In [ ]:
!mkdir -p /content/work

In [ ]:
%%writefile /content/work/stage1_topk.py
"""재정렬기 학습·검증에 쓸 1단계(FG-CLIP) 후보 목록을 만든다.

FG-CLIP 원격코드는 transformers==4.46.3 이 필요하고 재정렬기(Qwen3.5)는 5.x 가 필요하다. 그래서 이 스크립트는
4.46.3 을 별도 폴더에 설치해 PYTHONPATH 로 앞세워 따로 실행한다 (노트북 참고).

  PYTHONPATH=/content/tf446 python stage1_topk.py --stage1 <FT 폴더 | HF repo> --data /content/data/train_v1 \
      --val_ids val_ids.txt --out stage1_topk.json

출력 JSON
  train_topk: {학습 프롬프트 id: [학습 원본 이미지 id x k_train]}  재정렬기 학습용 '1단계가 헷갈린 오답' 후보
  val_topk:   {val 프롬프트 id: [val 후보 풀 이미지 id x k_val]}   재정렬 대상 (풀 = val 원본 500 + val HN)
  val_gt_rank:{val 프롬프트 id: 1단계 정답 순위(1부터, 동점 비관적)}
  stage1_val: 1단계 단독 val 점수, R@5/R@10
이미지 id 는 패키지 파일명(원본 '0005', HN '4005')이다.

test 모드 (--test, val_ids 불필요): 데이터 = test_v1, 후보 풀 = 전체 15,000장(원본 + HN), 질의 = 원본 프롬프트 8,000.
  test_topk:        {test 프롬프트 id: [이미지 id x k_test]}   재정렬 대상
  test_topk_scores: {test 프롬프트 id: [코사인 유사도 x k_test]}
  stage1_test:      1단계 단독 test 점수 (정답 = 같은 id, 서버 채점과 같은 규칙), R@5/R@10
"""
import argparse
import json
import math
from pathlib import Path

import torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset


class ImgDS(Dataset):
    def __init__(self, paths, proc):
        self.paths, self.proc = paths, proc

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        with Image.open(self.paths[i]) as im:
            return self.proc(images=im.convert("RGB"), return_tensors="pt")["pixel_values"][0]


def ensure_remote_code(stage1, base):
    """파인튜닝 폴더에 원격코드(*.py)가 없으면 원본 repo 에서 복사한다."""
    d = Path(stage1)
    if d.is_dir() and not list(d.glob("*.py")):
        from huggingface_hub import snapshot_download
        snap = Path(snapshot_download(base, allow_patterns=["*.py"]))
        for f in snap.glob("*.py"):
            (d / f.name).write_bytes(f.read_bytes())
        print(f"[stage1] {base} 원격코드를 {d} 로 복사")


def metrics(ranks):
    n = len(ranks)
    r1 = sum(r == 1 for r in ranks) / n
    nd = sum(1 / math.log2(r + 1) for r in ranks if r <= 5) / n
    return {"n": n, "recall_at_1": round(r1, 4), "ndcg_at_5": round(nd, 4), "score": round((r1 + nd) / 2, 4),
            "recall_at_5": round(sum(r <= 5 for r in ranks) / n, 4),
            "recall_at_10": round(sum(r <= 10 for r in ranks) / n, 4)}


def encode(model, tok, proc, dev, max_len, walk, img_paths, texts):
    """이미지·텍스트 정규화 임베딩 (float32, GPU)."""
    dl = DataLoader(ImgDS(img_paths, proc), batch_size=128, num_workers=4)
    feats, tfe = [], []
    with torch.inference_mode():
        for px in dl:
            feats.append(F.normalize(model.get_image_features(px.to(dev, torch.bfloat16)).float(), dim=-1))
        for s in range(0, len(texts), 256):
            t = tok(texts[s:s + 256], padding="max_length", truncation=True,
                    max_length=max_len, return_tensors="pt")["input_ids"].to(dev)
            tfe.append(F.normalize(model.get_text_features(t, walk_short_pos=walk).float(), dim=-1))
    return torch.cat(feats), torch.cat(tfe)


def run_test(a, model, tok, proc, dev, max_len, walk, text, orig, hn, files):
    import time
    ids = orig + hn
    print(f"[stage1-test] {a.stage1} | 질의 {len(orig)} / 후보 풀 {len(ids)}")
    t0 = time.time()
    img, txt = encode(model, tok, proc, dev, max_len, walk, [files[i] for i in ids], [text[i] for i in orig])
    t_enc = time.time() - t0
    S = txt @ img.T                                            # (질의, 풀)
    top_s, top_i = S.topk(a.k_test, dim=1)
    pos = {i: j for j, i in enumerate(ids)}
    gt = S[torch.arange(len(orig)), torch.tensor([pos[q] for q in orig])]
    ranks = (S >= gt[:, None]).sum(1).tolist()                 # 정답 = 같은 id, 비관적 순위
    res = {"stage1": a.stage1, "mode": "test", "k_test": a.k_test,
           "test_topk": {q: [ids[j] for j in row] for q, row in zip(orig, top_i.tolist())},
           "test_topk_scores": {q: [round(x, 6) for x in row] for q, row in zip(orig, top_s.tolist())},
           "stage1_test": metrics(ranks), "encode_sec": round(t_enc, 1)}
    Path(a.out).write_text(json.dumps(res))
    print("[stage1-test] 1단계 단독 test:", res["stage1_test"], f"| encode {t_enc:.0f}s")
    print("->", a.out)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--stage1", required=True, help="FG-CLIP 파인튜닝 폴더 또는 HF repo id")
    ap.add_argument("--base", default="qihoo360/fg-clip-base")
    ap.add_argument("--data", required=True, help="train_v1 폴더 (prompts.jsonl, images/)")
    ap.add_argument("--val_ids", help="train 모드에서 필수")
    ap.add_argument("--test", action="store_true", help="test_v1 전체 풀에서 원본 프롬프트별 top-k")
    ap.add_argument("--k_test", type=int, default=10)
    ap.add_argument("--out", required=True)
    ap.add_argument("--k_train", type=int, default=20)
    ap.add_argument("--k_val", type=int, default=20)
    ap.add_argument("--long", action="store_true", help="248토큰 모드")
    a = ap.parse_args()

    from transformers import AutoImageProcessor, AutoModelForCausalLM, AutoTokenizer
    ensure_remote_code(a.stage1, a.base)
    dev = torch.device("cuda")
    model = AutoModelForCausalLM.from_pretrained(a.stage1, trust_remote_code=True, torch_dtype=torch.bfloat16).to(dev).eval()
    tok = AutoTokenizer.from_pretrained(a.stage1, trust_remote_code=True)
    proc = AutoImageProcessor.from_pretrained(a.stage1, trust_remote_code=True)
    max_len, walk = (248, False) if a.long else (77, True)

    data = Path(a.data)
    rows = [json.loads(l) for l in open(data / "prompts.jsonl", encoding="utf-8") if l.strip()]
    text = {r["id"]: r.get("prompts", r.get("prompt")) for r in rows}
    orig = sorted(i for i, t in text.items() if t != "hard_negative")
    hn = sorted(i for i, t in text.items() if t == "hard_negative")
    width = len(orig[0])
    hn_src = {h: f"{int(h) - len(orig):0{width}d}" for h in hn}       # chan_lee pair_hn.py 와 같은 순서 규칙
    files = {p.stem: p for p in (data / "images").rglob("*.png")}
    if a.test:
        return run_test(a, model, tok, proc, dev, max_len, walk, text, orig, hn, files)
    val = {f"{int(l):0{width}d}" for l in open(a.val_ids) if l.strip()}
    fit = [i for i in orig if i not in val]
    val = sorted(val)
    val_pool = val + [h for h in hn if hn_src[h] in set(val)]
    print(f"[stage1] {a.stage1} | fit {len(fit)} / val {len(val)} / val pool {len(val_pool)}")

    ids = orig + hn
    img_t, txt_t = encode(model, tok, proc, dev, max_len, walk, [files[i] for i in ids], [text[i] for i in orig])
    img = dict(zip(ids, img_t))
    txt = dict(zip(orig, txt_t))

    def topk(queries, pool, k):
        P = torch.stack([img[i] for i in pool])
        out, ranks = {}, {}
        for q in queries:
            s = P @ txt[q]
            out[q] = [pool[j] for j in s.topk(min(k, len(pool))).indices.tolist()]
            if q in pool:
                ranks[q] = int((s >= s[pool.index(q)]).sum().item())   # 정답 이상 점수 개수 = 비관적 순위
        return out, ranks

    train_topk, _ = topk(fit, fit, a.k_train)
    val_topk, val_rank = topk(val, val_pool, a.k_val)
    res = {"stage1": a.stage1, "k_train": a.k_train, "k_val": a.k_val, "val_pool": val_pool,
           "train_topk": train_topk, "val_topk": val_topk, "val_gt_rank": val_rank,
           "stage1_val": metrics([val_rank[q] for q in val])}
    Path(a.out).write_text(json.dumps(res))
    print("[stage1] val (1단계 단독):", res["stage1_val"])
    print("->", a.out)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/work/rerank_lib.py
"""VLM Yes/No 재정렬기: zero-shot 채점, LoRA 파인튜닝, val 평가.

채점은 chan_lee rerankers/vlm_yesno.py 와 같다: 같은 질문 문구, enable_thinking=False,
score = logsumexp(logit[Yes, yes, ' Yes']) - logsumexp(logit[No, no, ' No']) (마지막 위치).

파인튜닝 (LoRA, 비전 인코더 고정):
  프롬프트 하나마다 [정답 이미지 1 + 오답 N_NEG] 묶음을 만들고, 묶음 안의 Yes-No 점수로
  정답이 1등이 되도록 softmax cross-entropy(listwise)를 건다. 추론은 쌍마다 독립 채점 그대로다.
  오답 후보: 같은 stage1 묶음의 형제 원본 + 1단계(FG-CLIP) top-k 오답. use_hn=True 면 자기 train HN 이미지를
  (있을 때) 오답 1개로 반드시 넣는다. 묶음 크기는 HN 사용 여부와 무관하게 같다.
"""
from __future__ import annotations

import json
import math
import random
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import torch
import torch.nn.functional as F
from PIL import Image

QUESTION = ('Does this image accurately match the following description, including every '
            'object count, color, material, pattern, pose and spatial relation?\n'
            'Description: "{prompt}"\nAnswer with Yes or No.')

_POOL = ThreadPoolExecutor(8)


def load_image(p):
    with Image.open(p) as im:
        return im.convert("RGB")


# ------------------------------------------------------------------ data
class TrainData:
    """train_v1 패키지 + val_ids + 번들 표 + 1단계 후보(stage1_topk.json)."""

    def __init__(self, data_dir, val_ids_path, bundles_path, stage1_json):
        data = Path(data_dir)
        rows = [json.loads(l) for l in open(data / "prompts.jsonl", encoding="utf-8") if l.strip()]
        text = {r["id"]: r.get("prompts", r.get("prompt")) for r in rows}
        self.orig = sorted(i for i, t in text.items() if t != "hard_negative")
        hn = sorted(i for i, t in text.items() if t == "hard_negative")
        w = len(self.orig[0])
        self.norm = lambda x: f"{int(x):0{w}d}"
        self.prompt = {i: text[i] for i in self.orig}
        self.path = {p.stem: p for p in (data / "images").rglob("*.png")}
        self.hn_of = {f"{int(h) - len(self.orig):0{w}d}": h for h in hn}           # 원본 id -> 자기 HN id
        val = {self.norm(l) for l in open(val_ids_path) if l.strip()}
        self.val = sorted(val)
        self.fit = [i for i in self.orig if i not in val]
        b = {self.norm(k): v for k, v in json.load(open(bundles_path)).items()}
        self.siblings = {}
        groups = {}
        for i in self.fit:
            if b.get(i):
                groups.setdefault(b[i], []).append(i)
        for g in groups.values():
            for i in g:
                self.siblings[i] = [j for j in g if j != i]
        s1 = json.load(open(stage1_json))
        self.stage1 = s1
        self.train_topk = s1["train_topk"]
        self.val_topk = s1["val_topk"]
        self.val_gt_rank = s1["val_gt_rank"]
        n_hn_fit = sum(i in self.hn_of for i in self.fit)
        print(f"[data] fit {len(self.fit)} (HN 있는 문항 {n_hn_fit}) / val {len(self.val)} | "
              f"형제 있는 fit 문항 {len(self.siblings)}")

    def negatives(self, pid, n_neg, use_hn, rng, s1_pool=10):
        negs = [self.hn_of[pid]] if (use_hn and pid in self.hn_of) else []
        cand = list(dict.fromkeys(self.siblings.get(pid, []) +
                                  [j for j in self.train_topk[pid][:s1_pool + 1] if j != pid]))
        rng.shuffle(cand)
        return negs + cand[:n_neg - len(negs)]


# ------------------------------------------------------------------ model
class Reranker:
    def __init__(self, model_id, max_pixels=448 * 448, adapter=None, lora=None, grad_ckpt=True):
        from transformers import AutoModelForImageTextToText, AutoProcessor
        self.dev = torch.device("cuda")
        self.model = AutoModelForImageTextToText.from_pretrained(model_id, dtype=torch.bfloat16).to(self.dev)
        self.proc = AutoProcessor.from_pretrained(model_id, min_pixels=64 * 64, max_pixels=max_pixels)
        self.proc.tokenizer.padding_side = "left"
        tok = self.proc.tokenizer
        self.yes = [tok.encode(w, add_special_tokens=False)[0] for w in ("Yes", "yes", " Yes")]
        self.no = [tok.encode(w, add_special_tokens=False)[0] for w in ("No", "no", " No")]
        if adapter:
            from peft import PeftModel
            self.model = PeftModel.from_pretrained(self.model, adapter)
            print(f"[rerank] LoRA 어댑터 로드: {adapter}")
        elif lora:
            self._add_lora(**lora, grad_ckpt=grad_ckpt)
        self.model.eval()

    def _add_lora(self, r=16, alpha=32, dropout=0.05, grad_ckpt=True):
        """언어 모델의 Linear 층에만 LoRA (비전 인코더·lm_head 제외). 대상 층 이름은 실제 모델에서 찾는다."""
        from peft import LoraConfig, get_peft_model
        suffixes = set()
        for name, mod in self.model.named_modules():
            if isinstance(mod, torch.nn.Linear) and not any(k in name for k in ("visual", "vision", "lm_head")):
                suffixes.add(name.rsplit(".", 1)[-1])
        print(f"[lora] 대상 Linear 이름: {sorted(suffixes)}")
        pattern = r"^(?!.*(visual|vision)).*\.(" + "|".join(sorted(suffixes)) + r")$"
        cfg = LoraConfig(r=r, lora_alpha=alpha, lora_dropout=dropout, target_modules=pattern, bias="none")
        if grad_ckpt:
            self.model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
            self.model.enable_input_require_grads()
        self.model = get_peft_model(self.model, cfg)
        for p in self.model.parameters():
            if p.requires_grad:
                p.data = p.data.float()                 # LoRA 가중치는 fp32 로 학습
        self.model.print_trainable_parameters()

    def chat(self, prompt):
        msgs = [{"role": "user", "content": [{"type": "image"},
                                             {"type": "text", "text": QUESTION.format(prompt=prompt)}]}]
        return self.proc.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False, enable_thinking=False)

    def margins(self, prompts, images):
        """(프롬프트, 이미지) 쌍마다 Yes-No 점수. 학습 중이면 그래디언트가 흐른다."""
        inputs = self.proc(text=[self.chat(p) for p in prompts], images=images, return_tensors="pt",
                           padding=True).to(self.dev)
        with torch.autocast("cuda", dtype=torch.bfloat16):
            logits = self.model(**inputs, logits_to_keep=1).logits[:, -1].float()
        return torch.logsumexp(logits[:, self.yes], -1) - torch.logsumexp(logits[:, self.no], -1)

    def save(self, out):
        self.model.save_pretrained(out)


# ------------------------------------------------------------------ eval
def score_metrics(ranks):
    n = len(ranks)
    r1 = sum(r == 1 for r in ranks) / n
    nd = sum(1 / math.log2(r + 1) for r in ranks if r <= 5) / n
    return {"n": n, "recall_at_1": round(r1, 4), "ndcg_at_5": round(nd, 4), "score": round((r1 + nd) / 2, 4)}


def rerank_ranks(cands, gt, gt_stage1_rank, scores):
    """후보 cands(1단계 순) 를 scores 로 재정렬했을 때 정답 순위. 정답이 후보 밖이면 1단계 순위 그대로.
    동점은 비관적(정답과 점수가 같은 다른 후보를 위로)."""
    if gt not in cands:
        return gt_stage1_rank
    g = scores[cands.index(gt)]
    return 1 + sum(1 for c, s in zip(cands, scores) if c != gt and s >= g)


@torch.no_grad()
def evaluate(rr: Reranker, data: TrainData, ks=(5, 10), batch=10, limit=None):
    rr.model.eval()
    kmax = max(ks)
    qs = data.val[:limit] if limit else data.val
    out = {k: [] for k in ks}
    t0, n_pairs = time.time(), 0
    for qi, q in enumerate(qs):
        cands = data.val_topk[q][:kmax]
        ims = list(_POOL.map(load_image, [data.path[c] for c in cands]))
        sc = []
        for s in range(0, len(cands), batch):
            sc += rr.margins([data.prompt[q]] * len(ims[s:s + batch]), ims[s:s + batch]).tolist()
        n_pairs += len(cands)
        for k in ks:
            out[k].append(rerank_ranks(cands[:k], q, data.val_gt_rank[q], sc[:k]))
        if qi % 100 == 0:
            print(f"  [eval] {qi}/{len(qs)}  {time.time() - t0:.0f}s", flush=True)
    sec = time.time() - t0
    res = {f"k{k}": score_metrics(out[k]) for k in ks}
    res["stage1"] = score_metrics([data.val_gt_rank[q] for q in qs])
    res["sec_per_pair"] = round(sec / n_pairs, 4)
    res["eval_sec"] = round(sec, 1)
    return res


# ------------------------------------------------------------------ train
def train(rr: Reranker, data: TrainData, *, use_hn, n_neg=3, prompts_per_step=2, grad_accum=4, epochs=1,
          lr=1e-4, warmup=0.05, max_prompts=None, seed=0, log_every=20):
    rng = random.Random(seed)
    params = [p for p in rr.model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.0)
    n_prompts = min(len(data.fit), max_prompts or len(data.fit))
    steps = epochs * math.ceil(n_prompts / (prompts_per_step * grad_accum))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min(1.0, (s + 1) / max(1, int(warmup * steps))) * 0.5 * (1 + math.cos(math.pi * min(s, steps) / steps)))
    log, t0, opt_step = [], time.time(), 0
    for ep in range(epochs):
        rr.model.train()
        order = data.fit[:]
        rng.shuffle(order)
        order = order[:n_prompts]
        run_loss, run_acc = [], []
        for i in range(0, len(order), prompts_per_step):
            group = order[i:i + prompts_per_step]
            prompts, paths, sizes = [], [], []
            for q in group:
                cands = [q] + data.negatives(q, n_neg, use_hn, rng)
                prompts += [data.prompt[q]] * len(cands)
                paths += [data.path[c] for c in cands]
                sizes.append(len(cands))
            ims = list(_POOL.map(load_image, paths))
            m = rr.margins(prompts, ims)
            # 묶음별 softmax CE (정답이 각 묶음의 0번)
            loss, acc, s = 0, 0, 0
            for sz in sizes:
                g = m[s:s + sz]
                loss = loss + F.cross_entropy(g[None], torch.zeros(1, dtype=torch.long, device=g.device))
                acc += int(len(g) == 1 or bool(g[0] > g[1:].max()))    # 동점은 오답 처리
                s += sz
            loss = loss / len(sizes)
            (loss / grad_accum).backward()
            run_loss.append(loss.item()); run_acc.append(acc / len(sizes))
            if (i // prompts_per_step + 1) % grad_accum == 0 or i + prompts_per_step >= len(order):
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                opt.step(); sched.step(); opt.zero_grad(set_to_none=True)
                opt_step += 1
                if opt_step % log_every == 0:
                    el = time.time() - t0
                    done = (ep * len(order) + i + len(group)) / (epochs * len(order))
                    print(f"[ep {ep}] step {opt_step}/{steps} loss {sum(run_loss[-log_every * grad_accum:]) / len(run_loss[-log_every * grad_accum:]):.4f} "
                          f"acc {sum(run_acc[-log_every * grad_accum:]) / len(run_acc[-log_every * grad_accum:]):.3f} "
                          f"lr {sched.get_last_lr()[0]:.2e} | {el / 60:.1f}분, 남은 시간 약 {el / done * (1 - done) / 60:.0f}분", flush=True)
        log.append({"epoch": ep, "loss": sum(run_loss) / len(run_loss), "acc": sum(run_acc) / len(run_acc)})
    rr.model.eval()
    return {"epochs": log, "train_min": round((time.time() - t0) / 60, 1), "opt_steps": opt_step}


# ------------------------------------------------------------------ test
@torch.no_grad()
def rerank_test(rr: Reranker, data_dir, topk_json, out_csv, *, k=5, pairs_per_batch=10, ckpt=None,
                public=range(7000, 8000), log_every=500):
    """test 원본 프롬프트마다 1단계 top-k 를 재정렬해 서버 형식 CSV(prompt_id,prediction 상위 5개)를 쓴다.

    ckpt(jsonl): 문항별 재정렬 점수를 이어 쓰고, 다시 실행하면 끝난 문항은 건너뛴다 (Colab 끊김 대비).
    동점은 1단계 순서를 유지한다(예측이므로 비관적 처리 없음). 채점은 정답 = 같은 id (서버 규칙과 같음).
    """
    data = Path(data_dir)
    rows = [json.loads(l) for l in open(data / "prompts.jsonl", encoding="utf-8") if l.strip()]
    prompt = {r["id"]: r.get("prompts", r.get("prompt")) for r in rows if r.get("prompts", r.get("prompt")) != "hard_negative"}
    path = {p.stem: p for p in (data / "images").rglob("*.png")}
    s1 = json.load(open(topk_json))
    topk = s1["test_topk"]
    qs = sorted(prompt)
    assert len(topk[qs[0]]) >= k, f"1단계 후보가 {len(topk[qs[0]])}개뿐 (k={k})"

    done = {}
    if ckpt and Path(ckpt).exists():
        for l in open(ckpt):
            if l.strip():
                d = json.loads(l)
                if d["k"] == k:
                    done[d["id"]] = d["s"]
    todo = [q for q in qs if q not in done]
    print(f"[test] 문항 {len(qs)} / 이미 끝남 {len(done)} / 남음 {len(todo)} | K={k}, 쌍 {len(todo) * k}")

    q_per = max(1, pairs_per_batch // k)
    chunks = [todo[i:i + q_per] for i in range(0, len(todo), q_per)]
    load = lambda ch: [_POOL.submit(load_image, path[c]) for q in ch for c in topk[q][:k]]
    t0, buf = time.time(), []
    fh = open(ckpt, "a") if ckpt else None
    nxt = load(chunks[0]) if chunks else None
    for ci, ch in enumerate(chunks):
        ims = [f.result() for f in nxt]
        if ci + 1 < len(chunks):
            nxt = load(chunks[ci + 1])                       # 다음 묶음 이미지를 미리 읽음
        sc = rr.margins([prompt[q] for q in ch for _ in range(k)], ims).tolist()
        for j, q in enumerate(ch):
            done[q] = [round(x, 5) for x in sc[j * k:(j + 1) * k]]
            buf.append(json.dumps({"id": q, "k": k, "s": done[q]}))
        if fh and (len(buf) >= 50 or ci + 1 == len(chunks)):
            fh.write("\n".join(buf) + "\n"); fh.flush(); buf = []
        n = (ci + 1) * q_per
        if n % log_every < q_per:
            el = time.time() - t0
            print(f"  [test] {min(n, len(todo))}/{len(todo)}  {el / 60:.1f}분, 남은 시간 약 "
                  f"{el / min(n, len(todo)) * (len(todo) - min(n, len(todo))) / 60:.0f}분", flush=True)
    if fh:
        fh.close()
    sec = time.time() - t0

    ranks, ranks_s1, pub = [], [], []
    with open(out_csv, "w") as f:
        f.write("prompt_id,prediction\n")
        for q in qs:
            c = topk[q]
            order = sorted(range(k), key=lambda j: (-done[q][j], j))
            pred = ([c[j] for j in order] + c[k:])[:5]
            f.write(f"{q},{' '.join(pred)}\n")
            ranks.append(pred.index(q) + 1 if q in pred else 99)
            ranks_s1.append(c[:5].index(q) + 1 if q in c[:5] else 99)
            pub.append(int(q) in public)
    res = {"k": k, "n": len(qs),
           "rerank": score_metrics(ranks),
           "rerank_public": score_metrics([r for r, p in zip(ranks, pub) if p]),
           "rerank_private": score_metrics([r for r, p in zip(ranks, pub) if not p]),
           "stage1_top5": score_metrics(ranks_s1),
           "rerank_sec_this_run": round(sec, 1),
           "sec_per_pair_this_run": round(sec / max(1, len(todo) * k), 4),
           "csv": str(out_csv)}
    print(json.dumps(res, indent=1))
    return res

## 6. 1단계 후보 만들기 (FG-CLIP, 한 번만)

train 3,500문항마다 1단계 top-20(학습용 오답 후보)과 val 500문항마다 top-20(재정렬 대상)을 저장한다. 마지막 줄에 1단계 단독 val 점수와 R@5/R@10(재정렬 상한)이 나온다.

In [ ]:
STAGE1_JSON = f"{RERANK_DIR}/stage1_topk.json"
if not os.path.exists(STAGE1_JSON):
    !PYTHONPATH=/content/tf446 python /content/work/stage1_topk.py --stage1 "{STAGE1}" --data "{DATA_DIR}" \
        --val_ids "{VAL_IDS_PATH}" --out "{STAGE1_JSON}"
print(json.load(open(STAGE1_JSON))["stage1_val"])

In [ ]:
import sys, gc
sys.path.insert(0, "/content/work")
import importlib, rerank_lib as R; importlib.reload(R)
data = R.TrainData(DATA_DIR, VAL_IDS_PATH, BUNDLES_PATH, STAGE1_JSON)

def run_eval(rr, name):
    p = f"{RERANK_DIR}/results_{name}.json"
    res = R.evaluate(rr, data, ks=KS, limit=EVAL_LIMIT)
    res.update({"name": name, "reranker": RERANKER, "stage1": STAGE1, "max_pixels": MAX_PIXELS})
    json.dump(res, open(p, "w"), indent=1)
    print(name, json.dumps({k: v for k, v in res.items() if k.startswith("k") or k == "stage1"}, indent=1))
    print("쌍당", res["sec_per_pair"], "s")
    return res

def free(rr):
    del rr; gc.collect(); torch.cuda.empty_cache()

def run_ft(name, use_hn):
    ad = f"{RERANK_DIR}/adapter_{name}"
    if os.path.exists(f"{ad}/adapter_config.json"):
        print("어댑터 있음 -> 학습 건너뜀:", ad)
    else:
        rr = R.Reranker(RERANKER, MAX_PIXELS, lora=LORA)
        log = R.train(rr, data, use_hn=use_hn, n_neg=N_NEG, prompts_per_step=PROMPTS_PER_STEP,
                      grad_accum=GRAD_ACCUM, epochs=EPOCHS, lr=LR, max_prompts=MAX_PROMPTS, seed=SEED)
        rr.save(ad)
        json.dump(log | {"use_hn": use_hn, "n_neg": N_NEG, "lr": LR, "lora": LORA, "epochs": EPOCHS},
                  open(f"{ad}/train_log.json", "w"), indent=1)
        print("학습", log["train_min"], "분 ->", ad)
        free(rr)
    rr = R.Reranker(RERANKER, MAX_PIXELS, adapter=ad)
    res = run_eval(rr, name)
    free(rr)
    return res

## 7. R0: zero-shot 재정렬 (chan_lee 방식)

In [ ]:
if not os.path.exists(f"{RERANK_DIR}/results_R0.json"):
    rr = R.Reranker(RERANKER, MAX_PIXELS)
    run_eval(rr, "R0"); free(rr)
else:
    print(json.load(open(f"{RERANK_DIR}/results_R0.json")))

## 8. R1: LoRA 파인튜닝 (형제 + 1단계 오답)

로그의 `acc`는 학습 묶음(정답 1 + 오답 3)에서 정답이 1등인 비율이다. 처음 0.25 근처에서 올라가야 정상이다.

In [ ]:
run_ft("R1", use_hn=False)

## 9. R2: LoRA 파인튜닝 + train HN 이미지

In [ ]:
run_ft("R2", use_hn=True)

## 10. 비교 (val 500, 묶음 val + HN 102장)

In [ ]:
import pandas as pd
rows = []
for n in ["R0", "R1", "R2"]:
    p = f"{RERANK_DIR}/results_{n}.json"
    if not os.path.exists(p): continue
    r = json.load(open(p))
    if not rows:
        rows.append({"조건": "1단계 단독", "score": r["stage1"]["score"], "R@1": r["stage1"]["recall_at_1"]})
    for k in KS:
        rows.append({"조건": f"{n} K={k}", "score": r[f"k{k}"]["score"], "R@1": r[f"k{k}"]["recall_at_1"],
                     "쌍당 초": r["sec_per_pair"]})
display(pd.DataFrame(rows))
print("1단계 R@5 / R@10 (K=5 / K=10 의 상한):", {k: json.load(open(STAGE1_JSON))["stage1_val"][k] for k in ("recall_at_5", "recall_at_10")})

## 11. test 재정렬 (test_v1 8,000문항, 후보 15,000장)

1단계(FG-CLIP)로 test 전체 풀에서 문항별 top-10을 뽑고, 그중 top-`TEST_K`를 LoRA 재정렬기로 다시 정렬해 서버 형식 CSV(`prompt_id,prediction` 상위 5개)를 만든다.

- 필요한 Drive 파일: `test_v1.tar` (공유 폴더에서 내 드라이브로 바로가기). **test 프롬프트·이미지·정답은 출력하지 않는다.**
- 시간(L4, 2B, 쌍당 약 0.067초): K=5면 4만 쌍으로 약 45분, K=10이면 약 90분.
- 문항별 점수를 Drive `test_scores_*.jsonl`에 이어 쓴다. **런타임이 끊기면 1~5와 이 셀들을 다시 실행하면 끝난 문항은 건너뛴다.**
- 마지막에 1단계 top-5 그대로의 점수(`stage1_top5`)도 같이 나온다. 이게 Modal 결과 0.6731과 거의 같으면 1단계가 제대로 돈 것이다.

In [ ]:
TEST_ADAPTER = "R1"     # 쓸 LoRA 어댑터 (RERANK_DIR/adapter_<이름>). zero-shot 이면 None
TEST_K = 5              # 재정렬할 1단계 후보 수
TEST_TAR = ""           # 비우면 MyDrive 에서 test_v1.tar 자동 탐색

TEST_ROOT = "/content/data_test"          # train 데이터(/content/data)와 섞이지 않게 따로 푼다
if not glob.glob(f"{TEST_ROOT}/**/prompts.jsonl", recursive=True):
    TEST_TAR = TEST_TAR or find_in_drive("test_v1.tar")
    os.makedirs(TEST_ROOT, exist_ok=True)
    !cp "{TEST_TAR}" /content/test_v1.tar
    !tar -xf /content/test_v1.tar -C "{TEST_ROOT}" --no-same-owner
    !rm /content/test_v1.tar
TEST_DIR = os.path.dirname(glob.glob(f"{TEST_ROOT}/**/prompts.jsonl", recursive=True)[0])
print("TEST_DIR =", TEST_DIR, "| png:", sum(1 for _ in glob.iglob(TEST_DIR + "/images/**/*.png", recursive=True)), "(15000 이어야 함)")

STAGE1_TEST_JSON = f"{RERANK_DIR}/stage1_test_topk.json"
if not os.path.exists(STAGE1_TEST_JSON):
    !PYTHONPATH=/content/tf446 python /content/work/stage1_topk.py --test --k_test 10 --stage1 "{STAGE1}" \
        --data "{TEST_DIR}" --out "{STAGE1_TEST_JSON}"
print("1단계 단독 test:", json.load(open(STAGE1_TEST_JSON))["stage1_test"])

In [ ]:
import sys, gc
sys.path.insert(0, "/content/work")
import importlib, rerank_lib as R; importlib.reload(R)
tag = f"{TEST_ADAPTER or 'R0'}_k{TEST_K}"
ad = f"{RERANK_DIR}/adapter_{TEST_ADAPTER}" if TEST_ADAPTER else None
rr = R.Reranker(RERANKER, MAX_PIXELS, adapter=ad)
test_res = R.rerank_test(rr, TEST_DIR, STAGE1_TEST_JSON, f"{RERANK_DIR}/test_{tag}.csv", k=TEST_K,
                         ckpt=f"{RERANK_DIR}/test_scores_{tag}.jsonl")
test_res.update({"adapter": ad, "reranker": RERANKER, "stage1": STAGE1, "max_pixels": MAX_PIXELS,
                 "stage1_test_full": json.load(open(STAGE1_TEST_JSON))["stage1_test"],
                 "gpu": torch.cuda.get_device_name(0), "peak_vram_gb": round(torch.cuda.max_memory_allocated() / 2**30, 2)})
json.dump(test_res, open(f"{RERANK_DIR}/test_results_{tag}.json", "w"), indent=1)
print("->", f"{RERANK_DIR}/test_results_{tag}.json")